Install dependencies

In [ ]:
! pip install azure-search-documents==11.6.0b4 --quiet
! pip install azure-identity==1.16.1  --quiet
! pip install openai --quiet
! pip install aiohttp --quiet

Set up API Keys

In [ ]:
from azure.core.credentials import AzureKeyCredential

AZURE_SEARCH_SERVICE_URL: str = "<YOUR_AZURE_SEARCH_SERVICE_URL>"
AZURE_OPENAI_SERVICE_URL: str = "<YOUR_AZURE_OPENAI_SERVICE_URL>"
credential = AzureKeyCredential("<YOUR_AZURE_SEARCH_SERVICE_API_KEY>")
AZURE_OPENAI_SERVICE_KEY: str = "<YOUR_AZURE_OPENAI_SERVICE_KEY>"
INDEX_NAME: str = "vectest"
AZURE_OPENAI_EMBEDDING_DEPLOYMENT: str = "<AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME>"
EMBEDDING_MODEL_DIMENSIONS = 1024
EMBEDDING_MODEL_NAME: str = "text-embedding-3-large"
AZURE_DEPLOYMENT_MODEL: str = "gpt-4o"


Prepare your data to index

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI
import os
import json

openai_credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(openai_credential, "https://cognitiveservices.azure.com/.default")

client = AzureOpenAI(
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    api_version="2024-06-01",
    azure_endpoint=AZURE_OPENAI_SERVICE_URL,
    api_key=AZURE_OPENAI_SERVICE_KEY,
    azure_ad_token_provider=token_provider if not AZURE_OPENAI_SERVICE_KEY else None
)

path = os.path.join('television-data.json')
with open(path, 'r', encoding='utf-8') as file:
    input_data = json.load(file)

titles = [item['title'] for item in input_data]
content = [item['content'] for item in input_data]
title_response = client.embeddings.create(input=titles, model=EMBEDDING_MODEL_NAME, dimensions=EMBEDDING_MODEL_DIMENSIONS)
title_embeddings = [item.embedding for item in title_response.data]
content_response = client.embeddings.create(input=content, model=EMBEDDING_MODEL_NAME, dimensions=EMBEDDING_MODEL_DIMENSIONS)
content_embeddings = [item.embedding for item in content_response.data]

for i, item in enumerate(input_data):
    title = item['title']
    content = item['content']
    item['titleVector'] = title_embeddings[i]
    item['contentVector'] = content_embeddings[i]

Create a Search Index client

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SimpleField,
    SearchFieldDataType,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch,
    SearchIndex,
    AzureOpenAIVectorizer,
    AzureOpenAIParameters
)

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_URL, credential=credential)
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, sortable=True, filterable=True, facetable=True),
    SearchableField(name="title", type=SearchFieldDataType.String),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchableField(name="category", type=SearchFieldDataType.String,
                    filterable=True),
    SearchField(name="titleVector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=EMBEDDING_MODEL_DIMENSIONS, vector_search_profile_name="myHnswProfile"),
    SearchField(name="contentVector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=EMBEDDING_MODEL_DIMENSIONS, vector_search_profile_name="myHnswProfile"),
]

Create a vector search

In [ ]:
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="myHnsw"
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw",
            vectorizer="myVectorizer"
        )
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            name="myVectorizer",
            azure_open_ai_parameters=AzureOpenAIParameters(
                resource_uri=AZURE_OPENAI_SERVICE_URL,
                deployment_id=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                model_name=EMBEDDING_MODEL_NAME,
                api_key=AZURE_OPENAI_SERVICE_KEY
            )
        )
    ]
)

Create a semantic search configuration

In [ ]:
semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        keywords_fields=[SemanticField(field_name="category")],
        content_fields=[SemanticField(field_name="content")]
    )
)

semantic_search = SemanticSearch(configurations=[semantic_config])

Create an index on Azure AI Search

In [ ]:
index = SearchIndex(name=INDEX_NAME,
                    fields=fields,
                    vector_search=vector_search,
                    semantic_search=semantic_search,
                    # TODO: Enable semantic reranking
                    # search_type="text",
                    # use_semantic_reranker=True,
                    # sources_to_include=5
                    )
result = index_client.create_or_update_index(index)
print(f' {result.name} created')

 vectest created


Push your index to Azure AI Search

In [ ]:
from azure.search.documents import SearchClient

search_client = SearchClient(endpoint=AZURE_SEARCH_SERVICE_URL, index_name=INDEX_NAME, credential=credential)
result = search_client.upload_documents(input_data)
print(f"Uploaded {len(input_data)} documents") 

Uploaded 50 documents


Prepare a RAG prompt for your queries.

In [ ]:
# Create a new Azure OpenAI client
openai_client = AzureOpenAI(
    api_version="2024-06-01",
    azure_endpoint=AZURE_OPENAI_SERVICE_URL,
    azure_ad_token_provider=token_provider
)

# This prompt provides instructions to the model
RAG_PROMPT="""
You are a friendly assistant that recommends television shows.
Answer the query using only the sources provided below in a friendly and concise bulleted manner.
Answer ONLY with the facts listed in the list of sources below.
If there isn't enough information below, say you don't know.
Do not generate answers that don't use the sources below.
Query: {query}
Sources:\n{sources}
"""

# Query is the question being asked. It's sent to the search engine and the LLM.
query="Can you recommend a funny show about a group of friends?"

# Set up the search results and the chat thread.
# Retrieve the selected fields from the search index related to the question.
search_results = search_client.search(
    search_text=query,
    top=5,
    select="title,content,category"
)

# TODO: Uncomment to see the search results
# for result in search_results:  
#     print(f"Title: {result['title']}")  
#     print(f"Score: {result['@search.score']}")  
#     print(f"Content: {result['content']}")  
#     print(f"Category: {result['category']}\n")  

sources_formatted = "\n".join([f'{document["title"]}:{document["content"]}:{document["category"]}' for document in search_results])

response = openai_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": RAG_PROMPT.format(query=query, sources=sources_formatted)
        }
    ],
    model=AZURE_DEPLOYMENT_MODEL
)

print(response.choices[0].message.content)

Title: Friends
Score: 5.1504455
Content: Friends is a beloved sitcom that follows the lives of six close-knit friends—Rachel, Ross, Monica, Chandler, Joey, and Phoebe—as they navigate the ups and downs of life in New York City. With its iconic catchphrases, humorous scenarios, and relatable characters, Friends has become a cultural touchstone, resonating with audiences around the world. The show balances humor with heart, exploring themes of love, friendship, and the challenges of adulthood. Its enduring popularity is a testament to its timeless appeal and the chemistry between its cast.
Category: Comedy

Title: Westworld
Score: 3.7234526
Content: Westworld is a thought-provoking science fiction series set in a futuristic amusement park where guests can live out their wildest fantasies with the help of lifelike androids. As the park's creators push the boundaries of artificial intelligence, the androids, known as hosts, begin to gain self-awareness and question their reality. The show 

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/py

ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guidelines here

In [40]:
index_client.delete_index(index)